In [2]:
import os
from  dotenv import load_dotenv
load_dotenv()
from langchain_community.document_loaders import PyMuPDFLoader

In [3]:
attention_file_path = r"doc\attention.pdf"  #- scan pdf not working

In [4]:
from langchain_community.document_loaders.parsers import RapidOCRBlobParser
loader = PyMuPDFLoader(
    file_path=attention_file_path,
    mode="page",
    extract_tables="markdown",
    extract_images=True,
    images_parser=RapidOCRBlobParser(),
    images_inner_format="markdown-img",
)

docs = loader.load()

Consider using the pymupdf_layout package for a greatly improved page layout analysis.


In [11]:
print(docs[7].page_content)

Table 2: The Transformer achieves better BLEU scores than previous state-of-the-art models on the
English-to-German and English-to-French newstest2014 tests at a fraction of the training cost.
Model
BLEU
Training Cost (FLOPs)
EN-DE
EN-FR
EN-DE
EN-FR
ByteNet [18]
23.75
Deep-Att + PosUnk [39]
39.2
1.0 · 1020
GNMT + RL [38]
24.6
39.92
2.3 · 1019
1.4 · 1020
ConvS2S [9]
25.16
40.46
9.6 · 1018
1.5 · 1020
MoE [32]
26.03
40.56
2.0 · 1019
1.2 · 1020
Deep-Att + PosUnk Ensemble [39]
40.4
8.0 · 1020
GNMT + RL Ensemble [38]
26.30
41.16
1.8 · 1020
1.1 · 1021
ConvS2S Ensemble [9]
26.36
41.29
7.7 · 1019
1.2 · 1021
Transformer (base model)
27.3
38.1
3.3 · 1018
Transformer (big)
28.4
41.8
2.3 · 1019
Residual Dropout
We apply dropout [33] to the output of each sub-layer, before it is added to the
sub-layer input and normalized. In addition, we apply dropout to the sums of the embeddings and the
positional encodings in both the encoder and decoder stacks. For the base model, we use a rate of
Pdrop = 0.1.


In [13]:
import fitz  #
from langchain_core.documents import Document

In [14]:
doc = fitz.open(attention_file_path)
image_dir = "images"
os.makedirs(image_dir, exist_ok=True)
image_docs = []

for page_index in range(len(doc)):
    page = doc[page_index]
    image_list = page.get_images()

    for image_index, img in enumerate(image_list, start=1):
        xref = img[0]
        pix = fitz.Pixmap(doc, xref)

        if pix.n - pix.alpha > 3:
            pix = fitz.Pixmap(fitz.csRGB, pix)

        img_path = f"{image_dir}/page_{page_index+1}_img_{image_index}.png"
        pix.save(img_path)

        # Store image reference as document
        image_docs.append(
            Document(
                page_content=f"Image on page {page_index+1}: {img_path}",
                metadata={"type": "image"}
            )
        )